# Setup the environment

### Windows

Download and install Cuda toolkit from:
https://developer.nvidia.com/cuda-downloads

Install torch and torchvision with the following command where cu124 is your Cuda compiler version (Use nvcc --version to check it)

In [ ]:
pip uninstall torchaudio torchvision torch -y

In [ ]:
pip install torch torchaudio torchvision --index-url https://download.pytorch.org/whl/cu124

In [ ]:
pip install -r requirements.txt

### Linux

Install Cuda toolkit with:

In [ ]:
sudo apt install nvidia-cuda-toolkit

In [ ]:
pip install torch torchvision

In [ ]:
pip install -r requirements.txt

# Load configuration
#### ALWAYS RUN BEFORE ANY EXECUTION.

MEAN and STD variables are updated in the .env file.

In [ ]:
# Description: Configuration file for the project

# DATA PATHS
RAW_DATA_PATH = 'dataset/raw/'
AUGMENTED_DATA_PATH = 'dataset/augmented/'
PROCESSED_DATA_TRAIN_PATH = 'dataset/processed/processed_data_train_exaltata.pt'
PROCESSED_DATA_VALIDATION_PATH = 'dataset/processed/processed_data_validation_exaltata.pt'
PROCESSED_DATA_TEST_PATH = 'dataset/processed/processed_data_test_exaltata.pt'
MODEL_PATH = 'models/model_exaltata.pt'

# Must be in the same order as the folders in the RAW_DATA_PATH
# CLASS_NAMES = ['O. exaltata', 'O. garganica', 'O. incubacea', 'O. sphegodes', 'O. sphegodes_Palena']
# CLASS_NAMES = ['O. exaltata', 'O. garganica', 'O. incubacea', 'O. majellensis', 'O. sphegodes', 'O. sphegodes_Palena']
CLASS_NAMES = ['O. exaltata', 'O. others']
# CLASS_NAMES = ['O. garganica', 'O. others']
# CLASS_NAMES = ['O. incubacea', 'O. others']
# CLASS_NAMES = ['O. majellensis', 'O. others']
# CLASS_NAMES = ['O. others', 'O. sphegodes']
# CLASS_NAMES = ['O. others', 'O. sphegodes_Palena']
# CLASS_NAMES = ['O. exaltata', 'O. exaltata_Int', 'O. garganica', 'O. garganica_Int', 'O. incubacea', 'O. incubacea_Int', 'O. majellensis', 'O. sphegodes', 'O. sphegodes_Int', 'O. sphegodes_Palena']
CLASS_SIZE = len(CLASS_NAMES)

# PREPROCESSING PARAMETERS (Run on CPU)
WIDTH = 256
HEIGHT = 512
TRAIN_SIZE = 0.7
VALIDATION_SIZE = 0.1 # TEST SIZE = 1 - TRAIN_SIZE - VALIDATION_SIZE
DYNAMIC_AUGMENTATION = False
BATCH_SIZE_PREPROCESS = 32 
NUM_WORKERS_PREPROCESS = 16 
SHOULD_NORMALIZE = True

# TRAINING PARAMETERS (Run on GPU)
FORCE_CPU = False
BATCH_SIZE_TRAIN = 8
NUM_WORKERS_TRAIN = 4
EPOCHS_TRAIN = 26
LEARNING_RATE = 0.001
PRETRAINED = True

# TESTING PARAMETERS
BATCH_SIZE_TEST = 32
NUM_WORKERS_TEST = 8
SLIDING_WINDOW_SIZE = 30
SLIDING_WINDOW_STRIDE = 15
DESTINATION_PATH = 'results/'

# INFERENCE PARAMETERS
PATH_TO_PREPROCESS = 'inference/raw/'
INFERENCE_PATH = 'inference/test/'
DESTINATION_PATH_INFERENCE = 'inference/results/'

# REPRODUCIBILITY
import random
import os
import torch
SEED = 57 #random.randint(0, 2**32 - 1)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8"  # Oppure ":4096:8"
os.environ["PYTHONHASHSEED"] = "0"
torch.use_deterministic_algorithms(True)#, warn_only=True)

# AUTOGENERATED PARAMETERS
MEAN='0.5069904327392578 0.5136597156524658 0.3467850387096405'
STD='0.21177464723587036 0.22732344269752502 0.20311179757118225'

Use the "%load .env" command to load the updated MEAN and STD variables

In [ ]:
# %load .env
MEAN='0.5069904327392578 0.5136597156524658 0.3467850387096405'
STD='0.21177464723587036 0.22732344269752502 0.20311179757118225'


# Preprocess

### Augmentation, splitting and resizing
The data is divided into the three train-validation-test datasets based on the required dimensions.

The functions defined in the "functions" array are applied to perform a dynamic augmentation in order to balance and increase the dataset. More transformations are applied to the smaller classes and fewer transformations to the larger ones. In any case, the identity transformation is always applied, so the final dataset could still be unbalanced.

It requires a work-folder so that the data can be renamed. This folder is automatically removed at the end of the operations.

In this phase, the photos are also resized and rotated so that the normalization and preprocessing can work with images that are all the same size.

In [ ]:
from app.preprocess_data import DynamicAugmentationWrapper
from app.preprocessing_tools import rotation
import shutil
import os

if __name__ == '__main__':
    workProcessedDataPath = 'work'
    functions = [rotation.identity, rotation.horizontal_and_vertical_flip, rotation.horizontal_flip, rotation.vertical_flip]
    DynamicAugmentationWrapper(RAW_DATA_PATH, workProcessedDataPath, AUGMENTED_DATA_PATH, CLASS_NAMES, TRAIN_SIZE, VALIDATION_SIZE, WIDTH, HEIGHT, DYNAMIC_AUGMENTATION, functions)
    print('Dynamic Augmentation Done')
    if os.path.exists(workProcessedDataPath):
        shutil.rmtree(workProcessedDataPath)
        print(f'Removed directory {workProcessedDataPath}')

### Refresh MEAN and STD

Updating the media requires that the images are all the same size, so the augmentation must be done first (possibly only with the "identity" function).

Running this code does not change the configuration cell but saves the updated information in the environment variable (which is lost when the notebook is closed) in the .env file

In [ ]:
from app.preprocess_data import fresh_normalization

if __name__ == '__main__':
    mean, std = fresh_normalization(AUGMENTED_DATA_PATH+"train/", WIDTH, HEIGHT, BATCH_SIZE_PREPROCESS, NUM_WORKERS_PREPROCESS, SEED)
    MEAN = ' '.join([str(x) for x in mean])
    STD = ' '.join([str(x) for x in std])
    print('MEAN:', MEAN)
    print('STD:', STD)

### Normalizzation and storing as tensors

In [ ]:
from app.preprocess_data import preprocessData

PATHS = [AUGMENTED_DATA_PATH+'train/', AUGMENTED_DATA_PATH+'validation/', AUGMENTED_DATA_PATH+'test/']

if __name__ == '__main__':
    print('Preprocessing training dataset...')
    preprocessData(PATHS[0], PROCESSED_DATA_TRAIN_PATH, WIDTH, HEIGHT, BATCH_SIZE_PREPROCESS, NUM_WORKERS_PREPROCESS, SHOULD_NORMALIZE, SEED, MEAN, STD)
    print('Preprocessing training dataset done')

    print('Preprocessing validation dataset...')
    preprocessData(PATHS[1], PROCESSED_DATA_VALIDATION_PATH, WIDTH, HEIGHT, BATCH_SIZE_PREPROCESS, NUM_WORKERS_PREPROCESS, SHOULD_NORMALIZE, SEED, MEAN, STD)
    print('Preprocessing validation dataset done')

    print('Preprocessing test dataset...')
    preprocessData(PATHS[2], PROCESSED_DATA_TEST_PATH, WIDTH, HEIGHT, BATCH_SIZE_PREPROCESS, NUM_WORKERS_PREPROCESS, SHOULD_NORMALIZE, SEED, MEAN, STD)
    print('Preprocessing test dataset done')
    print('Preprocessing done')




# Train

In [ ]:
import torch
from app.train_model import loadAndTrain, saveModel
if __name__ == '__main__':
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    if FORCE_CPU:
        device = torch.device('cpu')
    if device == torch.device('cpu'):
        print('WARNING: Running on CPU. Training will be slow.')
    model = loadAndTrain(device, CLASS_SIZE, PROCESSED_DATA_TRAIN_PATH, PROCESSED_DATA_VALIDATION_PATH, BATCH_SIZE_TRAIN, EPOCHS_TRAIN, NUM_WORKERS_TRAIN, LEARNING_RATE, PRETRAINED, MODEL_PATH, SEED)
    print('Training done.')
    saveModel(model, MODEL_PATH)

# Test

In [ ]:
from app.test_model import deviceLoader, modelLoader, getDatasetFromFile, testModel, generateOutputImages, showAndTestImages
from torch.utils.data import DataLoader
from app.preprocessing_tools.dataset_tool import getDatasetFromFile


if __name__ == '__main__':
    device = deviceLoader()
    model = modelLoader(MODEL_PATH, CLASS_SIZE, device)
    testDataset = getDatasetFromFile(PROCESSED_DATA_TEST_PATH)
    testDataLoader = DataLoader(testDataset, BATCH_SIZE_TEST, shuffle=True, num_workers=NUM_WORKERS_TEST, prefetch_factor=2, persistent_workers=True)
    testModel(model, testDataLoader, device, CLASS_NAMES, CLASS_SIZE)
    generateOutputImages(testDataset, model, device, CLASS_NAMES, DESTINATION_PATH, SLIDING_WINDOW_SIZE, SLIDING_WINDOW_STRIDE)
    # showAndTestImages(testDataset, model, device, CLASS_NAMES, SLIDING_WINDOW_SIZE, SLIDING_WINDOW_STRIDE) # Not working with notebook

### Inference

The following cell allows you to preprocess a single folder.


In [ ]:
from app.preprocessing_tools.dataset_tool import augmentDataPath
from app.preprocessing_tools import rotation
import os

if __name__ == '__main__':
    functions = [rotation.identity]
    if any(os.path.isdir(os.path.join(PATH_TO_PREPROCESS, folder)) for folder in os.listdir(PATH_TO_PREPROCESS)):
        for folder in os.listdir(PATH_TO_PREPROCESS):
            folder_path = os.path.join(PATH_TO_PREPROCESS, folder)
            if os.path.isdir(folder_path):
                augmentDataPath(folder_path, INFERENCE_PATH+folder+'/', 100000, WIDTH, HEIGHT, functions) 
    else:
        print('No images in the directory')
        augmentDataPath(PATH_TO_PREPROCESS, INFERENCE_PATH, 100000, WIDTH, HEIGHT, functions)

In [ ]:
from app.inference import loadDevice, loadModel, testInference, generateOutputImages, showAndTestImages
from app.preprocess_data import getTransforms
from app.preprocessing_tools.dataset_tool import SingleFolderDataset, ImageFolderWithName

if __name__ == '__main__':
    device = loadDevice(FORCE_CPU)
    model = loadModel(MODEL_PATH, CLASS_SIZE, device)
    model.eval()
    # test_dataset = ImageFolderWithName(
    #     root=INFERENCE_PATH,
    #     transform=getTransforms(WIDTH, HEIGHT, SHOULD_NORMALIZE, MEAN, STD),
    #     allow_empty=True
    # )
    test_dataset = SingleFolderDataset(
        INFERENCE_PATH,
        transform=getTransforms(WIDTH, HEIGHT, SHOULD_NORMALIZE, MEAN, STD),
    )
    # testInference(test_dataset, model, device, CLASS_NAMES)    # Require ImageFolderWithName dataset
    generateOutputImages(test_dataset, model, device, CLASS_NAMES, DESTINATION_PATH_INFERENCE, SLIDING_WINDOW_SIZE, SLIDING_WINDOW_STRIDE)
    # showAndTestImages(test_dataset, model, device, CLASS_NAMES, SLIDING_WINDOW_SIZE, SLIDING_WINDOW_STRIDE) # Not working with notebook

## 1VSALL

Test 1vsAll models

In [ ]:
from app.inference import loadDevice, loadModel, testInference1vsAll, inference1vsAll
from app.preprocess_data import getTransforms
from app.preprocessing_tools.dataset_tool import SingleFolderDataset, ImageFolderWithName

MODEL_PATHS = [f'models/1vallFullAugmented/{class_name}/model.pt' for class_name in CLASS_NAMES]

if __name__ == '__main__':
    device = loadDevice(FORCE_CPU)
    models = [loadModel(model_path, 2, device) for model_path in MODEL_PATHS]
    for model in models:
        model.eval()

    test_dataset = ImageFolderWithName(
        root=INFERENCE_PATH,
        transform=getTransforms(WIDTH, HEIGHT, SHOULD_NORMALIZE, MEAN, STD),
        allow_empty=True
    )
    testInference1vsAll(models, test_dataset, device, CLASS_NAMES)

Inference with 6 1vsAll models

In [ ]:
from app.inference import loadDevice, loadModel, testInference1vsAll, inference1vsAll
from app.preprocess_data import getTransforms
from app.preprocessing_tools.dataset_tool import SingleFolderDataset, ImageFolderWithName

MODEL_PATHS = [f'models/1vallFullAugmented/{class_name}/model.pt' for class_name in CLASS_NAMES]

if __name__ == '__main__':
    device = loadDevice(FORCE_CPU)
    models = [loadModel(model_path, 2, device) for model_path in MODEL_PATHS]
    for model in models:
        model.eval()
    test_dataset = SingleFolderDataset(
        INFERENCE_PATH,
        transform=getTransforms(WIDTH, HEIGHT, SHOULD_NORMALIZE, MEAN, STD),
    )
    for image, path in test_dataset:
        print(f'Inference for {path}')
        inference1vsAll(models, image.unsqueeze(0), device, 4)
        

### REST API for inference

In [ ]:
from flask import Flask, request, jsonify
import io
from PIL import Image
from app.inference import inference, loadDevice, loadModel
from app.preprocess_data import getTransforms

app = Flask(__name__)

@app.route('/inference', methods=['POST'])
def run_inference():
    if 'image' not in request.files:
        return jsonify({'error': 'No image provided'}), 400

    image_file = request.files['image']
    image = Image.open(io.BytesIO(image_file.read())).convert('RGB')
    transform = getTransforms(WIDTH, HEIGHT, False, MEAN, STD)
    image = transform(image).unsqueeze(0).to(device)

    values, predicted = inference(model, image, device)
    response = {
        'predicted_class': CLASS_NAMES[predicted.item()],
        'values': values.cpu().numpy().tolist()
    }
    return jsonify(response)

if __name__ == '__main__':
    device = loadDevice()
    model = loadModel(MODEL_PATH, len(CLASS_NAMES), device)
    app.run(host='0.0.0.0', port=5000)